# OMM 데이터 분석 — 1팀 (① 메뉴 군집 · ② 별점 × 재료비)

**사용법**
1. 맨 위 `[0단계] 준비` 셀부터 차례로 실행 (Shift+Enter)
2. 분석 코드를 고치려면 `%%writefile` 셀 안에서 고친 뒤 그 셀을 실행 → `report/analyses/` 의 내 파일이 저장돼요
3. 바로 아래 `[실행]` 셀을 실행하면 고친 코드로 그림이 다시 그려지고, `[데이터 확인]` 셀에서 불러온 DataFrame 을 볼 수 있어요
4. 사이트 반영: 관리자 화면 › 보고서 차트 › **다시 그리기** (또는 매일 18:30 자동). 데이터 분석 탭의 코드 칸은 저장하는 즉시 바뀌어요

> 다른 팀 파일(`analyses/` 의 다른 번호)과 `common.py` 는 고치지 마세요. 공통 부분을 바꿔야 하면 팀에 먼저 알려 주세요.

In [ ]:
# [0단계] 준비 — 공통 코드(common.py) 불러오기 · DB 접속 정보
import os, sys, importlib
REPORT = os.getenv('OMM_REPORT', '/opt/pipeline/api/report')   # back1 기준 분석 코드 폴더
sys.path.insert(0, REPORT)

# DB 접속: API 와 같은 읽기 전용 계정. 비밀번호는 화면에 출력하지 않아요
if not os.getenv('PGPASSWORD'):
    try:   # 설치 때 만든 읽기 전용 DB 접속 파일 (팀원 계정만 읽을 수 있음)
        for line in open(os.path.join(REPORT, '.pgenv'), encoding='utf-8'):
            k, _, v = line.strip().partition('=')
            if k.startswith('PG') and v: os.environ.setdefault(k, v.strip('"\''))
    except (FileNotFoundError, PermissionError):
        from getpass import getpass
        os.environ['PGPASSWORD'] = getpass('DB 비밀번호 (omm_api): ')

# 노트북에서 그린 그림은 notebooks/_out 에 저장 (사이트 그림은 건드리지 않음)
os.environ['REPORT_DIR'] = os.path.join(REPORT, 'notebooks', '_out')
os.makedirs(os.environ['REPORT_DIR'], exist_ok=True)

import common
common.SHOW = True          # save() 할 때 그림을 셀 아래에 바로 보여 주기
from common import *
print('준비 완료')

## ① 메뉴 군집
조리시간 · 1인분 재료비로 메뉴를 4유형으로 나누고, 상황에 맞는 메뉴 추천 기준으로 활용

- 파일: `report/analyses/a1_clusters.py` · 함수: `chart1()`
- 그림 파일: `notebooks/_out/` 에 PNG · SVG 로 저장

In [ ]:
%%writefile ../analyses/a1_clusters.py
"""① 메뉴 군집 — 조리시간 · 1인분 재료비로 메뉴를 4유형으로 나눔 → 상황별 추천 기준
담당: 1팀 (①②) · 이 파일만 고치면 돼요.
  - 노트북 notebooks/1팀_①②.ipynb 에서 고치고 바로 실행해 볼 수 있어요.
  - 서버는 매일 18:30 charts_v4.py 가 이 파일의 chart1() 을 불러 그림을 다시 그려요.
  - 관리자 화면 › 데이터 분석 탭의 '차트 그리기' 코드 칸에 이 파일 내용이 그대로 보여요.
"""
from common import *

# [1단계] DB에서 데이터 불러오기 (q = SQL → pandas DataFrame)
# [2단계] pandas 로 가공
# [3단계] matplotlib 으로 그리고 save() 로 저장 (PNG + SVG)

def chart1():
    d = q("""SELECT r.cook_time_min AS t, rm.cost_per_serving AS c, mc.name AS cl, mc.center_cook_min AS ct, mc.center_cost_won AS cc
             FROM omm_recipes r JOIN omm_recipe_metrics rm USING (recipe_id) JOIN omm_menu_clusters mc USING (cluster_id)
             WHERE rm.cost_per_serving > 0 AND r.cook_time_min > 0""")
    if d.empty: return empty('1_군집.png', 1, '조리시간 × 1인분 재료비 군집', 'mart_python_v4.py를 먼저 실행')
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [2.2, 1]})
    names = sorted(d.cl.unique(), key=lambda n: d[d.cl == n].ct.iloc[0] * 1e6 + d[d.cl == n].cc.iloc[0])
    rng = np.random.default_rng(0)
    for i, n in enumerate(names):
        g = d[d.cl == n]
        ax.scatter(g.t * np.exp(rng.normal(0, 0.06, len(g))), g.c, s=26, color=S[i], alpha=0.55, edgecolor=SURF, linewidth=0.6,
                   marker='ospD'[i % 4], label=f'{n} ({len(g):,})', rasterized=True)
        ax.scatter(g.ct.iloc[0], g.cc.iloc[0], s=260, marker='X', color=S[i], edgecolor=INK, linewidth=1.2, zorder=5)
        ax.annotate(n, (g.ct.iloc[0], g.cc.iloc[0]), xytext=(10, 8), textcoords='offset points', weight='bold', color=INK, fontsize=12)
    ax.set_xscale('log'); ax.set_yscale('log')
    logticks(ax.xaxis, [5, 10, 15, 20, 30, 60, 90, 120, 180], lambda v, p: f'{v:.0f}분')
    logticks(ax.yaxis, [300, 500, 1000, 2000, 3000, 5000, 10000, 20000], won); ax.set_xlabel('조리시간 (로그)'); ax.set_ylabel('1인분 재료비 (로그)')
    ax.set_title('① 조리시간 × 1인분 재료비 — 메뉴 4유형'); ax.legend(loc='upper left', fontsize=12)
    note(ax, '점 = 레시피 · X = 군집 중심 · K-means(로그 스케일 표준화) · 가격을 60% 이상 찾은 레시피만')
    cnt = d.cl.value_counts().reindex(names)
    ax2.barh(cnt.index, cnt.values, color=[S[i] for i in range(len(names))], height=0.6)
    for y, v in enumerate(cnt.values): ax2.text(v, y, f'  {v:,}', va='center', color=INK2)
    ax2.invert_yaxis(); ax2.set_title('유형별 레시피 수'); ax2.grid(axis='y', visible=False)
    save(fig, '1_군집.png', 1, '조리시간 × 재료비 군집', '메뉴를 자취형 · 대접형 등 4유형으로 나눈 결과')

if __name__ == '__main__':
    chart1()

In [ ]:
# [실행] 저장한 파일을 다시 읽어서 그림 그리기
from analyses import a1_clusters
importlib.reload(a1_clusters)
common.queries()          # 이전 기록 비우기
a1_clusters.chart1()

In [ ]:
# [데이터 확인] ① 메뉴 군집 이 DB 에서 불러온 DataFrame (SQL → pandas)
for i, (sql, df) in enumerate(common.queries(), 1):
    print(f'── 불러온 데이터 {i} · {len(df):,}행 × {df.shape[1]}열')
    display(df.head(10))

## ② 별점 × 재료비
별점과 예상 재료비의 상관관계 → 상관이 약하면 "저렴해도 만족도 높은 메뉴"를 추천 근거로 제시

- 파일: `report/analyses/a2_rating_cost.py` · 함수: `chart2()`
- 그림 파일: `notebooks/_out/` 에 PNG · SVG 로 저장

In [ ]:
%%writefile ../analyses/a2_rating_cost.py
"""② 별점 × 재료비 — 별점과 예상 재료비의 상관 → 약하면 "싸도 만족도 높은 메뉴" 추천 근거
담당: 1팀 (①②) · 이 파일만 고치면 돼요.
  - 노트북 notebooks/1팀_①②.ipynb 에서 고치고 바로 실행해 볼 수 있어요.
  - 서버는 매일 18:30 charts_v4.py 가 이 파일의 chart2() 을 불러 그림을 다시 그려요.
  - 관리자 화면 › 데이터 분석 탭의 '차트 그리기' 코드 칸에 이 파일 내용이 그대로 보여요.
"""
from common import *

# [1단계] DB에서 데이터 불러오기 (q = SQL → pandas DataFrame)
# [2단계] pandas 로 가공
# [3단계] matplotlib 으로 그리고 save() 로 저장 (PNG + SVG)

def chart2():
    sql = """SELECT r.title, r.avg_rating AS r, r.review_count AS n, rm.cost_per_serving AS c
             FROM omm_recipes r JOIN omm_recipe_metrics rm USING (recipe_id) WHERE r.review_count >= {} AND rm.cost_per_serving >= 300 AND rm.priced_ratio >= 0.6 AND r.avg_rating IS NOT NULL"""
    MINREV = 5
    d = q(sql.format(MINREV))
    if len(d) < 30: MINREV = 1; d = q(sql.format(MINREV))        # 최근 1년 레시피처럼 후기가 적으면 후기 1개 이상으로 낮춤
    if len(d) < 3: return empty('2_별점_재료비.png', 2, '별점 vs 재료비', '후기 있는 레시피가 부족')
    rr = np.corrcoef(np.log(d.c), d.r)[0, 1]
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw={'width_ratios': [1.3, 1], 'wspace': 0.75})
    ax.scatter(d.c, d.r, s=20 + np.clip(d.n, 5, 200) * 1.2, color=S[0], alpha=0.45, edgecolor=SURF, linewidth=0.6, rasterized=True)
    b = np.polyfit(np.log(d.c), d.r, 1); xs = np.linspace(np.log(d.c).min(), np.log(d.c).max(), 50)
    ax.plot(np.exp(xs), b[0] * xs + b[1], color=INK2, lw=2, ls='--')
    ax.set_xscale('log'); logticks(ax.xaxis, [300, 500, 1000, 2000, 3000, 5000, 10000, 20000], won); ax.set_xlabel('1인분 재료비 (로그)'); ax.set_ylabel('평균 별점')
    strength = '거의 없음' if abs(rr) < 0.1 else '약함' if abs(rr) < 0.3 else '보통' if abs(rr) < 0.5 else '강함'
    ax.set_title(f'② 별점 vs 1인분 재료비 (r = {rr:.2f}, {strength})')
    note(ax, f'점 크기 = 후기 수 · 점선 = 추세선 · 후기 {MINREV}개 이상 · 가격 붙은 재료 60% 이상 · 1인분 300원 이상 ({len(d):,}개) · 상관이 약하면 "싸도 만족도 높은 메뉴"를 추천할 근거')
    cut = d.c.quantile(0.25)
    top = d[d.c <= cut].sort_values(['r', 'n'], ascending=False).head(12)[::-1]
    ax2.barh([t[:12] + ('…' if len(t) > 12 else '') for t in top.title], top.r, color=S[2], height=0.6)
    for y, (v, c) in enumerate(zip(top.r, top.c)): ax2.text(v, y, f'  ★{v:.2f} · {c:,.0f}원', va='center', fontsize=11, color=INK2)
    ax2.set_xlim(max(0, top.r.min() - 0.6) if len(top) else 0, 5.6); ax2.set_title(f'싸도 만족도 높은 레시피\n(재료비 하위 25% · {cut:,.0f}원 이하)')
    ax2.grid(axis='y', visible=False)
    save(fig, '2_별점_재료비.png', 2, '별점 vs 재료비', f'상관 r = {rr:.2f}')

if __name__ == '__main__':
    chart2()

In [ ]:
# [실행] 저장한 파일을 다시 읽어서 그림 그리기
from analyses import a2_rating_cost
importlib.reload(a2_rating_cost)
common.queries()          # 이전 기록 비우기
a2_rating_cost.chart2()

In [ ]:
# [데이터 확인] ② 별점 × 재료비 이 DB 에서 불러온 DataFrame (SQL → pandas)
for i, (sql, df) in enumerate(common.queries(), 1):
    print(f'── 불러온 데이터 {i} · {len(df):,}행 × {df.shape[1]}열')
    display(df.head(10))

---
### 결과 정리 (직접 작성)
- 분석 결과:
- 서비스에 활용한 곳:
- 수익 모델과의 연결: